# 🎯 과제 3 (혼자 풀기): 페르소나를 JSON 파일로 분리

실무에서는 페르소나가 자주 추가·수정됩니다. 코드 안에 박혀 있으면 매번 코드를 고쳐야 하지만, **외부 파일로 분리하면 비개발자도 페르소나를 편집**할 수 있습니다.

## 목표
- `my_personas.json` 에 페르소나 정의를 저장한다
- 챗봇 시작 시 JSON을 읽어와 사용한다

## 진행 방식
1. `# TODO` 와 `...` 로 비어 있는 부분을 직접 채웁니다.
2. 각 단계 아래의 **✅ 확인 셀**을 실행해 통과(`🎉`)하면 다음 단계로 갑니다.
3. 막히면 **힌트** → 그래도 막히면 맨 아래 **정답 예시**를 보세요.

| 단계 | 할 일 |
|---|---|
| 0 | 환경 준비 (제공됨) |
| 1 | `my_personas.json` 만들기 |
| 2 | JSON 읽어오기 `load_personas()` |
| 3 | `ask()` 작성 |
| 4 | `handle_command()` 작성 |
| 5 | `run_chatbot()` 작성 및 실행 |
| 6 | 도전 과제 (선택) |

## 0단계. 환경 준비 (제공됨 — 그대로 실행)

In [1]:
import json
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()   # .env 의 OPENAI_API_KEY 로드
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

PERSONA_FILE = "my_personas.json"   # 이번 과제에서 만들 파일 (기존 personas.json 과 구분)

## 1단계. `my_personas.json` 만들기

**할 일**: 아래 셀의 `...` 를 채워 페르소나 3개 이상을 담은 딕셔너리를 만들고, 파일로 저장하세요.
- 키: 페르소나 이름 (예: `"분석가"`), 값: 시스템 프롬프트 문장
- 한글이 깨지지 않도록 `ensure_ascii=False`, `encoding="utf-8"` 를 꼭 지정!

> 💡 VS Code에서 직접 `my_personas.json` 파일을 만들어 편집해도 됩니다. 그 경우 이 셀은 건너뛰어도 OK.

<details><summary>힌트</summary>

```python
with open(PERSONA_FILE, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)
```
</details>

In [ ]:
# TODO: 페르소나 3개 이상 작성 (이름: 시스템 프롬프트)
initial_personas = {
    "분석가": "...",
    "마케터": "...",
    "개발자": "...",
}

# TODO: initial_personas 를 PERSONA_FILE 에 JSON 으로 저장
...

### ✅ 1단계 확인

In [ ]:
assert os.path.exists(PERSONA_FILE), f"{PERSONA_FILE} 파일이 없어요. 저장 코드를 확인하세요."
with open(PERSONA_FILE, encoding="utf-8") as f:
    _raw = f.read()
_data = json.loads(_raw)   # JSON 문법 오류가 있으면 여기서 에러
assert len(_data) >= 3, "페르소나는 3개 이상 필요해요."
assert all(v.strip() and v.strip() != "..." for v in _data.values()), "'...' 를 실제 프롬프트로 바꿔주세요."
assert "\\u" not in _raw, "한글이 \\uXXXX 로 저장됐어요. ensure_ascii=False 를 확인하세요."
print("🎉 1단계 통과! 페르소나:", list(_data.keys()))

## 2단계. JSON 읽어오기

**할 일**: 파일 경로를 받아 딕셔너리를 돌려주는 `load_personas(path)` 를 완성하세요.

<details><summary>힌트</summary>

```python
with open(path, encoding="utf-8") as f:
    return json.load(f)
```
</details>

In [ ]:
def load_personas(path: str) -> dict:
    # TODO: path 의 JSON 파일을 읽어 dict 로 반환
    ...

PERSONAS = load_personas(PERSONA_FILE)
print(PERSONAS)

### ✅ 2단계 확인

In [ ]:
assert isinstance(PERSONAS, dict), "load_personas 가 dict 를 반환해야 해요."
assert len(PERSONAS) >= 3
print("🎉 2단계 통과!")

## 3단계. `ask()` 작성

**할 일**: 페르소나 이름과 질문을 받아 LLM 답변 **문자열**을 반환하세요.

> ⚠️ 주의: JSON 의 프롬프트가 이미 `"당신은 ...입니다."` 형태라면, 템플릿에서 `"당신은 {role}입니다."` 로 감싸면 **"당신은 당신은 ..."** 처럼 중복됩니다. 아래 템플릿은 system 메시지를 **통째로** 받도록 만들어 두었어요.

흐름: ① `PERSONAS[이름]` 조회 → ② `chain.invoke({...})` → ③ `.content` 반환

<details><summary>힌트</summary>

`chain.invoke({"persona": ..., "question": ...})` 의 키는 템플릿의 `{변수}` 이름과 같아야 합니다.
</details>

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "{persona}"),
    ("human", "{question}"),
])
chain = prompt | llm   # LCEL

def ask(persona_name: str, question: str) -> str:
    # TODO 1: PERSONAS 에서 persona_name 에 해당하는 시스템 프롬프트 조회
    system_message = ...

    # TODO 2: chain 호출
    result = ...

    # TODO 3: AIMessage 에서 텍스트만 반환
    return ...

### ✅ 3단계 확인 (API 호출 1회)

In [ ]:
_name = list(PERSONAS.keys())[0]
_ans = ask(_name, "안녕! 한 문장으로 자기소개 해줘.")
assert isinstance(_ans, str) and _ans.strip(), "ask() 는 비어 있지 않은 문자열을 반환해야 해요."
print(f"[{_name}] {_ans}")
print("🎉 3단계 통과!")

## 4단계. `handle_command()` 작성

**요구사항** — `(새_페르소나, 종료여부)` 튜플을 반환합니다.

| 명령어 | 동작 |
|---|---|
| `/quit`, `/exit` | 종료 메시지 출력, 종료여부 `True` |
| `/persona` | 현재 페르소나 출력 |
| `/help` | 명령어 목록 출력 |
| `/change <이름>` | PERSONAS 에 있으면 변경, 없으면 사용 가능한 목록 안내 |
| 그 외 | 알 수 없는 명령어 안내 |

<details><summary>힌트</summary>

- `user_input.split(maxsplit=1)` → `"/change 마케터"` 가 `["/change", "마케터"]` 가 됩니다.
- `/change` 만 입력하면 `parts[1]` 에서 IndexError! → `parts[1] if len(parts) > 1 else ""`
</details>

In [ ]:
def handle_command(user_input: str, current_persona: str):
    parts = user_input.split(maxsplit=1)
    cmd = parts[0]

    # TODO: 위 표대로 명령어별 분기 작성
    if cmd in ("/quit", "/exit"):
        ...

    ...

    return current_persona, False

### ✅ 4단계 확인

In [ ]:
_names = list(PERSONAS.keys())
_a, _b = _names[0], _names[1]

assert handle_command("/quit", _a) == (_a, True), "/quit 은 (현재, True) 를 반환해야 해요."
assert handle_command("/exit", _a) == (_a, True)
assert handle_command("/persona", _a) == (_a, False)
assert handle_command("/help", _a) == (_a, False)
assert handle_command(f"/change {_b}", _a) == (_b, False), "/change 성공 시 새 페르소나를 반환해야 해요."
assert handle_command("/change 없는이름", _a) == (_a, False), "없는 이름이면 기존 유지!"
assert handle_command("/change", _a) == (_a, False), "/change 만 입력해도 에러 없이 처리되어야 해요."
assert handle_command("/zzz", _a) == (_a, False)
print("🎉 4단계 통과!")

## 5단계. `run_chatbot()` 작성 및 실행

**요구사항**
- 시작 시 인사말과 현재 페르소나 출력
- `while True` 로 `input()` 반복
- 빈 입력은 무시, `/` 로 시작하면 `handle_command`, 아니면 `ask`
- 종료 플래그가 `True` 면 루프 탈출
- 답변은 `[페르소나] 답변` 형태로 출력

<details><summary>힌트</summary>

```python
persona, should_quit = handle_command(user_input, persona)
if should_quit:
    break
```
</details>

In [ ]:
def run_chatbot(initial_persona: str = None):
    # 기본값: 파일의 첫 번째 페르소나
    persona = initial_persona or next(iter(PERSONAS))

    # TODO: 인사말 출력

    while True:
        # TODO: 입력 받기(strip), 빈 입력 무시, 명령어/질문 분기
        ...

# run_chatbot()   # ← 준비되면 주석 해제 후 실행. 입력창에 /quit 으로 종료

### ✅ 5단계 최종 점검 (직접 해보기)
`run_chatbot()` 을 실행하고 아래 시나리오를 모두 확인하세요.
- [ ] `/help` → 명령어 목록
- [ ] `/persona` → 현재 페르소나
- [ ] `/change <다른 이름>` → 변경 후 답변 말투가 달라짐
- [ ] `/change 없는이름` → 에러 없이 안내
- [ ] 빈 엔터 → 무시
- [ ] `/quit` → 종료

### 🔥 핵심 체험: 코드 수정 없이 페르소나 추가하기
1. `my_personas.json` 을 열어 새 페르소나(예: `"선생님": "당신은 친절한 초등학교 선생님입니다. ..."`)를 **파일에만** 추가하고 저장
2. 노트북 커널 재시작 없이 `PERSONAS = load_personas(PERSONA_FILE)` 만 다시 실행
3. `/change 선생님` 이 코드 수정 없이 동작하면 과제 성공! 🎉

## 6단계. 도전 과제 (선택)
1. **`/list` 명령어**: 등록된 페르소나 이름 전체 출력
2. **`/reload` 명령어**: 챗봇을 끄지 않고 JSON 을 다시 읽어오기 (`global PERSONAS` 또는 dict 갱신)
3. **예외 처리**: 파일이 없거나(`FileNotFoundError`) JSON 문법이 틀렸을 때(`json.JSONDecodeError`) 친절한 메시지를 출력하고 기본 페르소나로 대체
4. **`/add <이름> <프롬프트>`**: 대화 중 페르소나를 추가하고 JSON 파일에도 저장

In [ ]:
# 도전 과제 작업 공간

---
## 📝 정답 예시 (막힐 때만 펼쳐보세요)

<details><summary>핵심 코드 보기</summary>

```python
# 1단계
with open(PERSONA_FILE, "w", encoding="utf-8") as f:
    json.dump(initial_personas, f, ensure_ascii=False, indent=4)

# 2단계
def load_personas(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

# 3단계
def ask(persona_name, question):
    system_message = PERSONAS[persona_name]
    result = chain.invoke({"persona": system_message, "question": question})
    return result.content

# 4단계
def handle_command(user_input, current_persona):
    parts = user_input.split(maxsplit=1)
    cmd = parts[0]
    if cmd in ("/quit", "/exit"):
        print("👋 종료합니다.")
        return current_persona, True
    if cmd == "/persona":
        print(f"🎭 현재 페르소나: {current_persona}")
    elif cmd == "/help":
        print("📖 명령어: /quit, /exit, /change <이름>, /persona, /help")
    elif cmd == "/change":
        new_name = parts[1] if len(parts) > 1 else ""
        if new_name in PERSONAS:
            print(f"✅ 페르소나가 '{new_name}'(으)로 변경되었습니다.")
            return new_name, False
        print(f"❌ '{new_name}'은 등록된 페르소나가 아닙니다.")
        print(f"   사용 가능: {', '.join(PERSONAS)}")
    else:
        print(f"❓ 알 수 없는 명령어: {cmd} (/help 입력)")
    return current_persona, False

# 5단계
def run_chatbot(initial_persona=None):
    persona = initial_persona or next(iter(PERSONAS))
    print(f"👋 페르소나 챗봇입니다. (현재: {persona})")
    print("   /help 로 명령어 보기, /quit 로 종료\n")
    while True:
        user_input = input("질문> ").strip()
        if not user_input:
            continue
        if user_input.startswith("/"):
            persona, should_quit = handle_command(user_input, persona)
            if should_quit:
                break
        else:
            print(f"[{persona}] {ask(persona, user_input)}\n")
```
</details>